In [1]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    Model,
    CodeConfiguration,
    Environment
)
from azure.identity import DefaultAzureCredential
from pathlib import Path
import datetime

In [2]:
# Celda 2 - Conectar
ml_client = MLClient.from_config(credential=DefaultAzureCredential())
print(f"Conectado: {ml_client.workspace_name}")

Found the config file in: /config.json


Conectado: mlw-churn-dev


In [3]:
# Celda 3 - Crear endpoint
endpoint_name = "churn-asandoval-endpoint"

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Telco churn prediction endpoint",
    auth_mode="key",
)

ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f" Endpoint creado: {endpoint_name}")

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.8.1) and child packages mlflow-skinny (3.5.0) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()


 Endpoint creado: churn-asandoval-endpoint


In [6]:
# Celda 4 - Crear deployment
current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == 'notebooks' else current_dir

deployment = ManagedOnlineDeployment(
    name="churn-deployment",
    endpoint_name=endpoint_name,
    model=ml_client.models.get(name="telco-churn-model", version="3"),
    code_configuration=CodeConfiguration(
        code=str(project_root / "src"),
        scoring_script="score.py"
    ),
    environment=Environment(
        conda_file={
            "channels": ["conda-forge"],
            "dependencies": [
                "python=3.10",
                "pip",
                {
                    "pip": [
                        "numpy==1.23.5",
                        "scikit-learn==1.3.0",
                        "pandas",
                        "joblib",
                        "azureml-inference-server-http",
                    ]
                }
            ]
        },
        image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
    ),
    instance_type="Standard_DS2_v2",
    instance_count=1,
)

ml_client.online_deployments.begin_create_or_update(deployment).result()
print(f"Deployment creado")

Instance type Standard_DS2_v2 may be too small for compute resources. Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints. Learn more about SKUs here: https://learn.microsoft.com/azure/machine-learning/referencemanaged-online-endpoints-vm-sku-list
Check: endpoint churn-asandoval-endpoint exists


......................................................................

HttpResponseError: (BadArgument) User container has crashed or terminated. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-resourcenotready
Code: BadArgument
Message: User container has crashed or terminated. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-resourcenotready

In [5]:
ml_client.online_deployments.begin_delete(
    name="churn-deployment",
    endpoint_name=endpoint_name
).result()

In [12]:
# Celda 5 - Asignar 100% tráfico al deployment
endpoint.traffic = {"churn-deployment": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"✅ Tráfico asignado!")